# Bag of Words & TF-IDF


## Text Vectorization

ML models need **numbers**, not words. We convert text into numerical vectors.

| Method | Idea | Limitation |
|---|---|---|
| **Bag of Words** | Word count matrix | Ignores order & meaning |
| **TF-IDF** | Weighted word importance | Still no semantics |
| **Word Embeddings** | Dense semantic vectors | (→ Day 99) |

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

corpus = [
    "I love machine learning and deep learning",
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks for learning",
    "I love artificial intelligence and neural networks",
    "Natural language processing is part of AI"
]

print(f'Corpus: {len(corpus)} documents loaded')

## 1. Bag of Words (BoW)

In [ ]:
# BoW – CountVectorizer
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(corpus)

vocab = vectorizer.get_feature_names_out()
df_bow = pd.DataFrame(X_bow.toarray(), columns=vocab,
                       index=[f'Doc {i+1}' for i in range(len(corpus))])

print(f' BoW Matrix Shape: {X_bow.shape} ({X_bow.shape[0]} docs × {X_bow.shape[1]} words)\n')
print(df_bow.to_string())

In [ ]:
# Visualize BoW matrix
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(df_bow, annot=True, fmt='d', cmap='Blues', linewidths=0.5, ax=ax)
ax.set_title('Bag of Words Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Vocabulary')
ax.set_ylabel('Documents')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 2. TF-IDF (Term Frequency – Inverse Document Frequency)

$$\text{TF-IDF}(t, d) = \text{TF}(t,d) \times \text{IDF}(t)$$

- **TF** = how often term `t` appears in doc `d`  
- **IDF** = log(N / df(t)) — penalizes common words across all docs
- **High TF-IDF** = word is important in *this* doc but rare elsewhere

In [ ]:
# TF-IDF Vectorizer
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    X_tfidf.toarray().round(3),
    columns=tfidf.get_feature_names_out(),
    index=[f'Doc {i+1}' for i in range(len(corpus))]
)

print(f'TF-IDF Matrix Shape: {X_tfidf.shape}\n')
print(df_tfidf.to_string())

In [ ]:
# Visualize TF-IDF
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(df_tfidf, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('TF-IDF Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Vocabulary')
ax.set_ylabel('Documents')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. BoW vs TF-IDF Comparison

In [ ]:
# Top keywords per document
print('Top Keywords per Document (TF-IDF):\n')
for i, doc in enumerate(corpus):
    row = df_tfidf.iloc[i]
    top3 = row.nlargest(3)
    print(f'Doc {i+1}: "{doc[:45]}..."')
    for word, score in top3.items():
        print(f'        {word}: {score:.3f}')
    print()

## 4. N-grams – Capturing Phrases

In [ ]:
# Bigram + Unigram BoW
vec_ngram = CountVectorizer(ngram_range=(1, 2), max_features=20)
X_ngram = vec_ngram.fit_transform(corpus)

print('📊 N-gram Vocabulary (unigrams + bigrams):')
for feat in vec_ngram.get_feature_names_out():
    print(f'  • {feat}')

## 5. Text Classification with TF-IDF

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Larger dataset
docs = [
    "I love this movie great acting", "wonderful film loved every moment",
    "amazing story and direction", "excellent performances heartwarming",
    "brilliant masterpiece loved it", "terrible movie waste of time",
    "awful boring plot terrible acting", "worst film ever made",
    "horrible waste of money disappointing", "dreadful unwatchable garbage"
]
labels = [1,1,1,1,1, 0,0,0,0,0]  # 1=positive, 0=negative

X = TfidfVectorizer().fit_transform(docs)
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.3, random_state=42)

clf = LogisticRegression()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('🎯 Sentiment Classification Results:')
print(classification_report(y_test, y_pred, target_names=['Negative','Positive']))

## Summary

| Feature | BoW | TF-IDF |
|---|---|---|
| **Basis** | Word count | Weighted importance |
| **Common words** | High values | Penalized (low IDF) |
| **Rare words** | Low values | Rewarded (high IDF) |
| **Best for** | Simple tasks | Most NLP tasks |

